# CSA4704 – Deep Learning | Assignment 2 (CO2)
## Complete Google Colab experimental notebook

**Purpose:** Implement the seven questions in the supplied assignment brief, preserving its required experimental logic and evidence.

### Important dataset notes
- Q1 uses the public Breast Cancer Wisconsin dataset as a binary tabular benchmark. It is not a credit-card fraud dataset; the brief permits public tabular datasets.
- Q2 uses a clearly labelled **synthetic hospital-readmission-style dataset** because no approved patient dataset was supplied. It has the four requested feature types and is used only to verify NumPy backpropagation.
- Q3 uses the public scikit-learn Digits dataset (binary 0-vs-1 image task) as a small-image proxy. It is **not a plant-leaf dataset**; replace it with a faculty-approved leaf dataset if the context must be literal.
- Q4 downloads the public UCI Appliances Energy Prediction dataset.
- Q5 uses Fashion-MNIST.
- Q6 uses CIFAR-10 as a four-class small-image deployment benchmark (cat, deer, horse, dog). It is not a wildlife-camera-trap dataset; the limitation is stated rather than misrepresenting the classes.

### Reproducibility and execution
- Seeds are recorded and three runs are used where repeated-run variability is evaluated.
- Q4, Q5 and especially Q6 are computationally heavier. **Select Runtime → Change runtime type → T4 GPU** before running the image experiments. CPU remains supported but will take longer.
- Run cells in order. Do not report results until the cells have completed. The notebook exports raw CSVs and plots to `/content/CO2_outputs/`.
- No results are pre-filled or fabricated. Every reported metric is generated by execution.

## Assignment coverage checklist

| Question | Required evidence implemented |
|---|---|
| Q1 (6) | Predictions and derivatives; shallow/deep/poor-initialization edge case; sigmoid/tanh/ReLU; layer-wise early gradients; dead-ReLU fraction; loss; validation and test metrics; 3 seeds; 8-layer recommendation evidence |
| Q2 (4) | Computational graph; chain-rule equations; NumPy-only forward/backward/update; finite-difference checks for W1, b1, W2, b2 at two epsilons; deliberate bug and detection |
| Q3 (6) | Generalizing, severe-overfit and label-noise conditions; capacity/data-size predictions; L2, dropout, early stopping, augmentation and combination; curves, accuracy gap, best epoch, weight norm |
| Q4 (6) | SGD, Momentum, RMSProp, Adam; update rules; 3 learning rates × 2 batch sizes; target validation loss; epochs to target; RMSE/MAE; stability; 3 seeds |
| Q5 (7) | Student-designed CNN; output-size and parameter formulas; FC/CNN comparison; test accuracy/time/parameters; translated-test accuracy drop; learned filters and feature maps |
| Q6 (8) | Custom architecture family varying depth/width/resolution; parameters, MACs, file size, latency, accuracy; hand-verified layer shapes/parameters; growth predictions; deployment budget and trade-off |
| Q7 (13) | Diagnose a measured limitation; measurable objective; revised design; pre-training prediction; same-data/seed/budget validation; trade-offs and defence |

In [ ]:
# Cell 1 — environment, reproducibility, output folders
import os, sys, time, math, random, zipfile, urllib.request, copy, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from sklearn.datasets import load_breast_cancer, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, mean_absolute_error,
                             mean_squared_error)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, Subset
import torchvision
import torchvision.transforms as T

SEED_LIST = [42, 123, 2026]
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT = Path("/content/CO2_outputs")
OUT.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__, "| pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__, "| PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Device:", DEVICE)
print("Seeds:", SEED_LIST)
print("Outputs:", OUT)

# Q1 — Hidden units: Does the activation function matter as the network grows deeper? (6 marks)

**Prediction before implementation**
- Sigmoid has derivative `σ(z)(1−σ(z)) ≤ 0.25`; repeated multiplication through layers can attenuate early-layer gradients, especially when units saturate.
- Tanh has derivative `1−tanh²(z)`, which is also small for saturated activations but is centred around zero.
- ReLU has derivative 1 for positive pre-activations and 0 for negative pre-activations. It can preserve gradients along active paths but can create permanently inactive (“dead”) units.
- Xavier initialization is appropriate for sigmoid/tanh; He initialization is designed for ReLU. Deliberately large random weights are the edge case and may saturate units or destabilize activations.

**Configurations:** shallow `[64, 32]`; deep eight-layer `[64]×8`; edge case eight-layer `[64]×8` with deliberately large normal initialization. The same data split, optimizer, epochs and metrics are used across conditions.

In [ ]:
# Q1 — data: Breast Cancer Wisconsin (569 samples, 30 features)
q1 = load_breast_cancer()
X1, y1 = q1.data.astype(np.float32), q1.target.astype(np.int64)
X1_train, X1_tmp, y1_train, y1_tmp = train_test_split(
    X1, y1, test_size=0.30, random_state=SEED, stratify=y1)
X1_val, X1_test, y1_val, y1_test = train_test_split(
    X1_tmp, y1_tmp, test_size=0.50, random_state=SEED, stratify=y1_tmp)
sc1 = StandardScaler()
X1_train = sc1.fit_transform(X1_train).astype(np.float32)
X1_val = sc1.transform(X1_val).astype(np.float32)
X1_test = sc1.transform(X1_test).astype(np.float32)
print("Dataset:", q1.DESCR.splitlines()[0])
print("Samples/features:", X1.shape, "| classes:", q1.target_names)
print("Train/val/test:", len(y1_train), len(y1_val), len(y1_test))
print("Class counts:", np.bincount(y1_train), np.bincount(y1_val), np.bincount(y1_test))

In [ ]:
# Q1 — model, training, gradient norms and dead-ReLU measurement
Q1_CONFIGS = {
    "Shallow": [64, 32],
    "Deep_8L": [64]*8,
    "Edge_PoorInit_8L": [64]*8
}
Q1_EPOCHS = 80
Q1_LR = 1e-3

class Q1MLP(nn.Module):
    def __init__(self, dims, activation, poor_init=False):
        super().__init__()
        self.linears = nn.ModuleList()
        self.activation_name = activation
        prev = 30
        for width in dims:
            self.linears.append(nn.Linear(prev, width))
            prev = width
        self.out = nn.Linear(prev, 1)
        self.act = {"sigmoid": nn.Sigmoid, "tanh": nn.Tanh, "relu": nn.ReLU}[activation]()
        for layer in list(self.linears) + [self.out]:
            if poor_init:
                nn.init.normal_(layer.weight, mean=0.0, std=1.5)
            elif activation == "relu":
                nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
            else:
                nn.init.xavier_normal_(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(self, x, return_activations=False):
        acts, h = [], x
        for layer in self.linears:
            h = self.act(layer(h))
            acts.append(h)
        logits = self.out(h)
        return (logits, acts) if return_activations else logits

def q1_run(config, activation, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    dims = Q1_CONFIGS[config]
    model = Q1MLP(dims, activation, poor_init=(config=="Edge_PoorInit_8L")).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=Q1_LR)
    loss_fn = nn.BCEWithLogitsLoss()
    xt = torch.tensor(X1_train, device=DEVICE); yt = torch.tensor(y1_train[:,None].astype(np.float32), device=DEVICE)
    xv = torch.tensor(X1_val, device=DEVICE); yv = torch.tensor(y1_val[:,None].astype(np.float32), device=DEVICE)
    history, grad_rows = [], []
    for epoch in range(Q1_EPOCHS):
        model.train(); opt.zero_grad()
        logits = model(xt); loss = loss_fn(logits, yt); loss.backward()
        if epoch < 10:
            for li, layer in enumerate(model.linears):
                grad_rows.append({"config":config,"activation":activation,"seed":seed,
                                  "epoch":epoch+1,"layer":li+1,
                                  "gradient_norm":float(layer.weight.grad.norm().detach().cpu())})
        opt.step()
        model.eval()
        with torch.no_grad():
            tr_loss = loss_fn(model(xt), yt).item()
            va_logits = model(xv); va_loss = loss_fn(va_logits, yv).item()
            va_pred = (torch.sigmoid(va_logits).cpu().numpy().ravel() >= .5).astype(int)
        history.append({"epoch":epoch+1,"train_loss":tr_loss,"val_loss":va_loss,
                        "val_accuracy":accuracy_score(y1_val,va_pred)})
    # Dead ReLU fraction: fraction of hidden units whose activation is zero
    # for every training sample at the end of training.
    model.eval()
    with torch.no_grad():
        _, acts = model(xt, return_activations=True)
        dead = [float((a <= 0).all(dim=0).float().mean().cpu()) for a in acts] if activation=="relu" else [np.nan]*len(acts)
        test_logits = model(torch.tensor(X1_test,device=DEVICE))
        test_pred = (torch.sigmoid(test_logits).cpu().numpy().ravel() >= .5).astype(int)
    result = {
        "config":config,"activation":activation,"seed":seed,
        "test_accuracy":accuracy_score(y1_test,test_pred),
        "test_precision":precision_score(y1_test,test_pred,zero_division=0),
        "test_recall":recall_score(y1_test,test_pred,zero_division=0),
        "final_train_loss":history[-1]["train_loss"],
        "final_val_loss":history[-1]["val_loss"],
        "dead_relu_fraction_mean":float(np.nanmean(dead)) if activation=="relu" else np.nan,
        "history":history,"dead_by_layer":dead,"model":model
    }
    return result, grad_rows

q1_runs, q1_grad_rows = [], []
for cfg in Q1_CONFIGS:
    for act in ["sigmoid","tanh","relu"]:
        for seed in SEED_LIST:
            print("Q1:",cfg,act,"seed",seed)
            r,g = q1_run(cfg,act,seed)
            q1_runs.append(r); q1_grad_rows.extend(g)

q1_df = pd.DataFrame([{k:v for k,v in r.items() if k not in ["history","dead_by_layer","model"]} for r in q1_runs])
q1_summary = q1_df.groupby(["config","activation"]).agg(
    accuracy_mean=("test_accuracy","mean"), accuracy_std=("test_accuracy","std"),
    precision_mean=("test_precision","mean"), recall_mean=("test_recall","mean"),
    train_loss_mean=("final_train_loss","mean"), val_loss_mean=("final_val_loss","mean"),
    dead_relu_fraction=("dead_relu_fraction_mean","mean")).reset_index()
display(q1_summary.round(4))
q1_grad_df = pd.DataFrame(q1_grad_rows)
q1_grad_summary = q1_grad_df.groupby(["config","activation","layer","epoch"]).gradient_norm.agg(["mean","std"]).reset_index()
display(q1_grad_summary.head(20).round(6))
q1_df.to_csv(OUT/"Q1_runs.csv",index=False); q1_grad_df.to_csv(OUT/"Q1_gradient_norms.csv",index=False)
q1_summary.to_csv(OUT/"Q1_summary.csv",index=False)

In [ ]:
# Q1 — plots: loss curves and early-epoch layer-wise gradient norms
fig, axes = plt.subplots(1,3,figsize=(16,4))
for ax,cfg in zip(axes,Q1_CONFIGS):
    for act in ["sigmoid","tanh","relu"]:
        selected=[r for r in q1_runs if r["config"]==cfg and r["activation"]==act]
        tr=np.mean([[e["train_loss"] for e in r["history"]] for r in selected],axis=0)
        va=np.mean([[e["val_loss"] for e in r["history"]] for r in selected],axis=0)
        ax.plot(tr,label=f"{act} train"); ax.plot(va,"--",label=f"{act} val")
    ax.set_title(cfg); ax.set_xlabel("Epoch"); ax.set_ylabel("BCE loss"); ax.grid(True); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT/"Q1_loss_curves.png",dpi=160); plt.show()

gplot=q1_grad_df.groupby(["config","activation","layer"]).gradient_norm.mean().reset_index()
fig,axes=plt.subplots(1,3,figsize=(16,4))
for ax,cfg in zip(axes,Q1_CONFIGS):
    sub=gplot[gplot.config==cfg]
    for act in ["sigmoid","tanh","relu"]:
        z=sub[sub.activation==act]
        ax.plot(z.layer,z.gradient_norm,marker="o",label=act)
    ax.set_title(cfg+" (first 10 epochs)"); ax.set_xlabel("Hidden layer"); ax.set_ylabel("Mean gradient norm"); ax.set_yscale("symlog"); ax.grid(True); ax.legend()
plt.tight_layout(); plt.savefig(OUT/"Q1_gradient_by_layer.png",dpi=160); plt.show()

print("Dead ReLU fractions (mean across seeds and layers):")
display(q1_summary[q1_summary.activation=="relu"][["config","dead_relu_fraction"]].round(4))

### Q1 mathematical explanation and decision template

For layer `l`, backpropagation multiplies upstream gradients by the layer weight matrix and activation derivative:
`δ_l = (W_{l+1}ᵀ δ_{l+1}) ⊙ φ'(z_l)`.
The sigmoid derivative is `σ(z)(1−σ(z))`; tanh derivative is `1−tanh²(z)`; ReLU derivative is 1 for `z>0` and 0 for `z<0` (the value at zero is conventionally 0).

Use the generated gradient table to compare early and late layers. Do not claim vanishing gradients solely from accuracy. For the production constraint of at least eight hidden layers, select the activation/initialization using the **eight-layer** results, validation behaviour, gradient evidence and dead-ReLU fraction. The shallow winner need not be the deep winner.

# Q2 — Can you trust your gradients? Backpropagation from first principles (4 marks)

**Computational graph:** `X → Z1=XW1+b1 → A1=sigmoid(Z1) → Z2=A1W2+b2 → ŷ=sigmoid(Z2) → binary cross-entropy L`.

For a batch of `m` examples:
- `dZ2 = (ŷ−y)/m`
- `dW2 = A1ᵀ dZ2`, `db2 = sum(dZ2)`
- `dA1 = dZ2 W2ᵀ`
- `dZ1 = dA1 ⊙ A1(1−A1)`
- `dW1 = Xᵀ dZ1`, `db1 = sum(dZ1)`

The implementation below uses NumPy only; no PyTorch/TensorFlow autograd is used in Q2.

In [ ]:
# Q2 — synthetic hospital-readmission-style tabular data
# Explicitly synthetic: not real patient records.
rng=np.random.default_rng(2026)
n=1200
age=rng.normal(62,13,n).clip(18,95)
los=rng.gamma(2.2,2.0,n).clip(1,25)
prior=rng.poisson(1.4,n).clip(0,10)
meds=rng.poisson(5,n).clip(0,20)
X2=np.column_stack([age,los,prior,meds]).astype(float)
logit=-5.0+0.035*age+0.16*los+0.42*prior+0.07*meds
p=1/(1+np.exp(-logit))
y2=(rng.random(n)<p).astype(float)
X2tr,X2tmp,y2tr,y2tmp=train_test_split(X2,y2,test_size=.30,random_state=42,stratify=y2)
X2v,X2te,y2v,y2te=train_test_split(X2tmp,y2tmp,test_size=.50,random_state=42,stratify=y2tmp)
mu=X2tr.mean(0); sd=X2tr.std(0)+1e-8
X2tr=(X2tr-mu)/sd; X2v=(X2v-mu)/sd; X2te=(X2te-mu)/sd
print("Synthetic dataset:",X2.shape,"features: age, length of stay, prior admissions, medications")
print("Train/validation/test:",len(y2tr),len(y2v),len(y2te),"positive rate:",y2.mean().round(3))

In [ ]:
# Q2 — NumPy forward/backward/update and finite-difference gradient checking
def sigmoid_np(z):
    z=np.clip(z,-40,40)
    return 1/(1+np.exp(-z))

def init_np(d_in=4,d_hidden=5,seed=7):
    r=np.random.default_rng(seed)
    return {"W1":r.normal(0,.2,(d_in,d_hidden)),"b1":np.zeros((1,d_hidden)),
            "W2":r.normal(0,.2,(d_hidden,1)),"b2":np.zeros((1,1))}

def forward_np(X,p):
    z1=X@p["W1"]+p["b1"]; a1=sigmoid_np(z1)
    z2=a1@p["W2"]+p["b2"]; yhat=sigmoid_np(z2)
    return yhat,(X,z1,a1,z2)

def bce_np(yhat,y):
    eps=1e-12
    return -np.mean(y*np.log(yhat+eps)+(1-y)*np.log(1-yhat+eps))

def backward_np(yhat,y,cache,p,bug=False):
    X,z1,a1,z2=cache; m=len(y)
    dz2=(yhat-y.reshape(-1,1))/m
    dW2=a1.T@dz2; db2=dz2.sum(0,keepdims=True)
    da1=dz2@p["W2"].T
    dz1=da1 if bug else da1*a1*(1-a1)  # deliberate bug: omit sigmoid derivative
    dW1=X.T@dz1; db1=dz1.sum(0,keepdims=True)
    return {"W1":dW1,"b1":db1,"W2":dW2,"b2":db2}

def numerical_grads(X,y,p,eps):
    out={}
    for key in p:
        g=np.zeros_like(p[key])
        for idx in np.ndindex(p[key].shape):
            old=p[key][idx]
            p[key][idx]=old+eps; yp,_=forward_np(X,p); lp=bce_np(yp,y)
            p[key][idx]=old-eps; ym,_=forward_np(X,p); lm=bce_np(ym,y)
            p[key][idx]=old
            g[idx]=(lp-lm)/(2*eps)
        out[key]=g
    return out

def relative_error(a,n):
    return np.linalg.norm(a-n)/(np.linalg.norm(a)+np.linalg.norm(n)+1e-12)

Xgc=X2tr[:32]; ygc=y2tr[:32]
params=init_np(seed=11)
yp,cache=forward_np(Xgc,params)
analytic=backward_np(yp,ygc,cache,params)
gc_rows=[]
for eps in [1e-5,1e-6]:
    numeric=numerical_grads(Xgc,ygc,params,eps)
    for key in params:
        gc_rows.append({"epsilon":eps,"parameter":key,
                        "relative_error":relative_error(analytic[key],numeric[key])})
gc_df=pd.DataFrame(gc_rows)
print("Correct analytical gradients:")
display(gc_df.pivot(index="parameter",columns="epsilon",values="relative_error").applymap(lambda x:f"{x:.3e}"))

# Deliberate bug: omit sigmoid derivative in hidden layer
bug_grad=backward_np(yp,ygc,cache,params,bug=True)
bug_rows=[]
for key in params:
    bug_rows.append({"parameter":key,"relative_error":relative_error(bug_grad[key],numeric_grads(Xgc,ygc,params,1e-5)[key])})
print("Bugged backward-pass gradient errors (epsilon=1e-5):")
display(pd.DataFrame(bug_rows))

In [ ]:
# Q2 — train NumPy network with gradient descent and evaluate
params=init_np(seed=42)
lr=.1
q2_losses=[]
for epoch in range(1500):
    yh,ca=forward_np(X2tr,params)
    loss=bce_np(yh,y2tr)
    grads=backward_np(yh,y2tr,ca,params)
    for k in params: params[k]-=lr*grads[k]
    if epoch%100==0: q2_losses.append({"epoch":epoch,"loss":loss})
yh_test,_=forward_np(X2te,params)
pred=(yh_test.ravel()>=.5).astype(int)
print("Q2 test accuracy:",accuracy_score(y2te,pred))
print("Q2 test precision:",precision_score(y2te,pred,zero_division=0))
print("Q2 test recall:",recall_score(y2te,pred,zero_division=0))
display(pd.DataFrame(q2_losses))
pd.DataFrame(q2_losses).to_csv(OUT/"Q2_training_loss.csv",index=False)
gc_df.to_csv(OUT/"Q2_gradient_check.csv",index=False)

### Q2 acceptance criterion and bug interpretation

A relative error below approximately `1e−5` (and often much smaller for a well-scaled double-precision check) is strong evidence that analytical and numerical gradients agree. The exact tolerance depends on finite-difference step size, scale and floating-point error. The deliberate missing sigmoid derivative should primarily corrupt `W1` and `b1`; `W2` and `b2` are computed before the hidden derivative and should remain consistent. Use the actual displayed table to identify the first affected parameter group.

# Q3 — Engineer overfitting, then control it: Regularization (6 marks)

This section uses the public scikit-learn Digits dataset, restricted to digits 0 and 1 as a small binary image benchmark. This is a transparent proxy for the assignment's small labelled plant-leaf image scenario, not actual leaf imagery.

Conditions:
1. Generalization condition: larger training subset and a compact model.
2. Deliberate overfit: very small training subset and a high-capacity model.
3. Additional condition: label noise in the training labels.

The overfit condition compares L2 weight decay, dropout, early stopping, image translation augmentation, and a combination of L2 + dropout. Validation data is never augmented or used for gradient updates.

In [ ]:
# Q3 — load and prepare image data
digits=load_digits()
mask=np.isin(digits.target,[0,1])
X3=digits.images[mask].astype(np.float32)/16.0
y3=(digits.target[mask]==1).astype(np.int64)
X3=X3[:,None,:,:]
X3tr,X3tmp,y3tr,y3tmp=train_test_split(X3,y3,test_size=.30,random_state=42,stratify=y3)
X3v,X3te,y3v,y3te=train_test_split(X3tmp,y3tmp,test_size=.50,random_state=42,stratify=y3tmp)
print("Digits 0-vs-1 images:",X3.shape,"image shape:",X3.shape[1:])
print("Train/validation/test:",len(y3tr),len(y3v),len(y3te))

In [ ]:
# Q3 — MLP, training conditions and regularizers
class Q3MLP(nn.Module):
    def __init__(self,dropout=0.0,wide=True):
        super().__init__()
        h=256 if wide else 48
        self.net=nn.Sequential(nn.Flatten(),nn.Linear(64,h),nn.ReLU(),
                               nn.Dropout(dropout),nn.Linear(h,h//2),nn.ReLU(),
                               nn.Dropout(dropout),nn.Linear(h//2,1))
    def forward(self,x): return self.net(x)

def q3_train(Xtr,ytr, Xval,yval, method="none", seed=42, epochs=100, wide=True):
    torch.manual_seed(seed); np.random.seed(seed)
    model=Q3MLP(dropout=.5 if method in ["dropout","combined"] else 0,wide=wide).to(DEVICE)
    wd=1e-3 if method in ["l2","combined"] else 0
    opt=torch.optim.Adam(model.parameters(),lr=1e-3,weight_decay=wd)
    loss_fn=nn.BCEWithLogitsLoss()
    xt=torch.tensor(Xtr,device=DEVICE); yt=torch.tensor(ytr[:,None].astype(np.float32),device=DEVICE)
    xv=torch.tensor(Xval,device=DEVICE); yv=torch.tensor(yval[:,None].astype(np.float32),device=DEVICE)
    hist=[]; best_state=None; best_val=np.inf; best_epoch=0; patience=12; wait=0
    for ep in range(epochs):
        model.train()
        # Data augmentation: small random translations via affine_grid/grid_sample.
        if method=="augmentation":
            theta=torch.eye(2,3,device=DEVICE).unsqueeze(0).repeat(len(xt),1,1)
            theta[:,0,2]=torch.empty(len(xt),device=DEVICE).uniform_(-.12,.12)
            theta[:,1,2]=torch.empty(len(xt),device=DEVICE).uniform_(-.12,.12)
            grid=F.affine_grid(theta,xt.size(),align_corners=False)
            xb=F.grid_sample(xt,grid,align_corners=False)
        else: xb=xt
        opt.zero_grad(); logits=model(xb); loss=loss_fn(logits,yt); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            trp=(torch.sigmoid(model(xt)).cpu().numpy().ravel()>=.5).astype(int)
            vp=torch.sigmoid(model(xv)); vl=loss_fn(model(xv),yv).item()
            vpred=(vp.cpu().numpy().ravel()>=.5).astype(int)
            tl=loss_fn(model(xt),yt).item()
        hist.append({"epoch":ep+1,"train_loss":tl,"val_loss":vl,
                     "train_accuracy":accuracy_score(ytr,trp),
                     "val_accuracy":accuracy_score(yval,vpred)})
        if vl<best_val:
            best_val=vl; best_epoch=ep+1; best_state=copy.deepcopy(model.state_dict()); wait=0
        else: wait+=1
        if method=="early_stopping" and wait>=patience: break
    if best_state is not None: model.load_state_dict(best_state)
    with torch.no_grad():
        wnorm=float(torch.sqrt(sum((p**2).sum() for p in model.parameters())).cpu())
    return model,hist,best_epoch,wnorm

# Three deliberately different conditions; record predictions before training.
q3_conditions={
 "generalizing":{"fraction":.75,"wide":False,"noise":0.0,
  "prediction":"More data and lower capacity should reduce the train-validation gap."},
 "severe_overfit":{"fraction":.12,"wide":True,"noise":0.0,
  "prediction":"Small training set and high capacity should create a large train-validation gap."},
 "label_noise":{"fraction":.35,"wide":True,"noise":.20,
  "prediction":"Corrupting 20% of training labels should make fitting harder and may worsen generalization."}
}
print(json.dumps(q3_conditions,indent=2))

In [ ]:
# Q3 — evaluate three conditions and regularizers across 3 seeds
q3_rows=[]; q3_histories={}
for condition, spec in q3_conditions.items():
    for seed in SEED_LIST:
        rng=np.random.default_rng(seed)
        idx=np.arange(len(y3tr)); rng.shuffle(idx)
        idx=idx[:max(30,int(len(idx)*spec["fraction"]))]
        xx,yy=X3tr[idx].copy(),y3tr[idx].copy()
        if spec["noise"]>0:
            flip=rng.choice(len(yy),size=max(1,int(len(yy)*spec["noise"])),replace=False)
            yy[flip]=1-yy[flip]
        methods=["none"] if condition!="severe_overfit" else ["none","l2","dropout","early_stopping","augmentation","combined"]
        for method in methods:
            model,hist,best_epoch,wnorm=q3_train(xx,yy,X3v,y3v,method,seed,epochs=100,wide=spec["wide"])
            q3_histories[(condition,method,seed)]=hist
            q3_rows.append({"condition":condition,"method":method,"seed":seed,
                            "epochs_run":len(hist),"best_val_epoch":best_epoch,
                            "train_accuracy":hist[-1]["train_accuracy"],
                            "val_accuracy":hist[-1]["val_accuracy"],
                            "accuracy_gap":hist[-1]["train_accuracy"]-hist[-1]["val_accuracy"],
                            "min_val_loss":min(h["val_loss"] for h in hist),
                            "weight_norm":wnorm})
q3_df=pd.DataFrame(q3_rows)
display(q3_df.round(4))
q3_summary=q3_df.groupby(["condition","method"]).agg(
    val_accuracy_mean=("val_accuracy","mean"),gap_mean=("accuracy_gap","mean"),
    min_val_loss_mean=("min_val_loss","mean"),best_epoch_mean=("best_val_epoch","mean"),
    weight_norm_mean=("weight_norm","mean")).reset_index()
display(q3_summary.round(4))
q3_df.to_csv(OUT/"Q3_runs.csv",index=False); q3_summary.to_csv(OUT/"Q3_summary.csv",index=False)

In [ ]:
# Q3 — learning curves for the overfit condition and regularizer comparison
fig,axes=plt.subplots(1,2,figsize=(13,4))
for method in ["none","l2","dropout","early_stopping","augmentation","combined"]:
    hs=[q3_histories[("severe_overfit",method,s)] for s in SEED_LIST]
    maxlen=min(map(len,hs))
    tr=np.mean([[h[i]["train_loss"] for i in range(maxlen)] for h in hs],axis=0)
    va=np.mean([[h[i]["val_loss"] for i in range(maxlen)] for h in hs],axis=0)
    axes[0].plot(tr,label=method); axes[1].plot(va,label=method)
axes[0].set_title("Training loss"); axes[1].set_title("Validation loss")
for ax in axes: ax.set_xlabel("Epoch"); ax.set_ylabel("BCE loss"); ax.grid(True); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT/"Q3_regularization_curves.png",dpi=160); plt.show()

### Q3 interpretation prompts
Use the measured results to answer: (1) which condition produced the largest generalization gap; (2) which regularizer reduced the gap most; (3) whether the combined regularizers helped or over-regularized; (4) why the best validation-loss epoch may precede the final epoch; and (5) why training accuracy alone is insufficient. Do not assume a regularizer wins—state what your table actually shows.

# Q4 — Optimization for training deep models (6 marks)

Dataset: UCI Appliances Energy Prediction. Target: next-hour appliance energy (`Appliances`) using available weather, calendar/time and recent contextual features. The dataset is time ordered; the split is chronological to avoid using future records to predict earlier records.

Optimizer update rules (element-wise operations):
- SGD: `θₜ₊₁ = θₜ − ηgₜ`
- Momentum: `vₜ = βvₜ₋₁ + gₜ`; `θₜ₊₁ = θₜ − ηvₜ`
- RMSProp: `sₜ = ρsₜ₋₁ + (1−ρ)gₜ²`; `θₜ₊₁ = θₜ − ηgₜ/(√sₜ+ε)`
- Adam: `mₜ=β₁mₜ₋₁+(1−β₁)gₜ`; `vₜ=β₂vₜ₋₁+(1−β₂)gₜ²`; bias-correct `m̂,v̂`; `θₜ₊₁=θₜ−ηm̂/(√v̂+ε)`.

Prediction: adaptive optimizers (especially Adam) may reach a useful loss faster, while excessive learning rates can cause oscillation or divergence for all methods. This is a hypothesis to test, not a result.

In [ ]:
# Q4 — download and prepare UCI Appliances Energy Prediction dataset
import io
UCI_URL="https://archive.ics.uci.edu/static/public/374/appliances+energy+prediction.zip"
zip_path="/content/appliances_energy.zip"
urllib.request.urlretrieve(UCI_URL,zip_path)
with zipfile.ZipFile(zip_path) as z:
    csv_names=[n for n in z.namelist() if n.lower().endswith(".csv")]
    print("CSV files:",csv_names)
    raw=pd.read_csv(z.open(csv_names[0]))
print(raw.shape,raw.columns.tolist()[:10])
# Drop index-like column if present; convert date into time features.
if "date" in raw.columns:
    dt=pd.to_datetime(raw["date"],dayfirst=True,errors="coerce")
    raw["hour"]=dt.dt.hour; raw["dayofweek"]=dt.dt.dayofweek; raw["month"]=dt.dt.month
    raw=raw.drop(columns=["date"])
raw=raw.select_dtypes(include=[np.number]).dropna()
target="Appliances"
X4=raw.drop(columns=[target]).to_numpy(dtype=np.float32)
y4=raw[target].to_numpy(dtype=np.float32).reshape(-1,1)
n4=len(y4); a=int(.70*n4); b=int(.85*n4)
X4tr,X4v,X4te=X4[:a],X4[a:b],X4[b:]
y4tr,y4v,y4te=y4[:a],y4[a:b],y4[b:]
sc4=StandardScaler(); X4tr=sc4.fit_transform(X4tr).astype(np.float32)
X4v=sc4.transform(X4v).astype(np.float32); X4te=sc4.transform(X4te).astype(np.float32)
ys=y4tr.mean(); ystd=y4tr.std()+1e-8
y4tr_s=(y4tr-ys)/ystd; y4v_s=(y4v-ys)/ystd; y4te_s=(y4te-ys)/ystd
# Cap rows to a fixed chronological training budget to keep Colab runtime manageable.
MAX_TRAIN=6000
X4tr=X4tr[-MAX_TRAIN:]; y4tr_s=y4tr_s[-MAX_TRAIN:]
print("Dataset:",raw.shape,"features:",X4.shape[1],"chronological split:",len(X4tr),len(y4v),len(y4te))
print("Target units: Appliances energy (Wh); report RMSE/MAE in original units after inverse scaling.")

In [ ]:
# Q4 — same MLP, 4 optimizers × 3 learning rates × 2 batch sizes × 3 seeds
class Q4Regressor(nn.Module):
    def __init__(self,nin):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(nin,64),nn.ReLU(),nn.Linear(64,32),nn.ReLU(),nn.Linear(32,1))
    def forward(self,x): return self.net(x)

def make_optimizer(name,params,lr):
    if name=="SGD": return torch.optim.SGD(params,lr=lr)
    if name=="Momentum": return torch.optim.SGD(params,lr=lr,momentum=.9)
    if name=="RMSProp": return torch.optim.RMSprop(params,lr=lr,alpha=.99,eps=1e-8)
    if name=="Adam": return torch.optim.Adam(params,lr=lr)
    raise ValueError(name)

def q4_run(name,lr,batch,seed,epochs=20):
    torch.manual_seed(seed); np.random.seed(seed)
    model=Q4Regressor(X4tr.shape[1]).to(DEVICE)
    opt=make_optimizer(name,model.parameters(),lr)
    loss_fn=nn.MSELoss()
    ds=TensorDataset(torch.tensor(X4tr),torch.tensor(y4tr_s))
    gen=torch.Generator().manual_seed(seed)
    loader=DataLoader(ds,batch_size=batch,shuffle=True,generator=gen)
    xv=torch.tensor(X4v,device=DEVICE); yv=torch.tensor(y4v_s,device=DEVICE)
    history=[]; reached=None
    target_val_loss=.35  # chosen target in standardized-target MSE units
    for ep in range(epochs):
        model.train(); batch_losses=[]
        for xb,yb in loader:
            xb=xb.to(DEVICE); yb=yb.to(DEVICE)
            opt.zero_grad(); loss=loss_fn(model(xb),yb); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),10.0)
            opt.step(); batch_losses.append(loss.item())
        model.eval()
        with torch.no_grad(): vl=loss_fn(model(xv),yv).item()
        history.append({"epoch":ep+1,"train_loss":float(np.mean(batch_losses)),"val_loss":vl})
        if reached is None and vl<=target_val_loss: reached=ep+1
    with torch.no_grad():
        pred_s=model(torch.tensor(X4te,device=DEVICE)).cpu().numpy()
    pred=pred_s*ystd+ys
    rmse=float(np.sqrt(mean_squared_error(y4te,pred)))
    mae=float(mean_absolute_error(y4te,pred))
    vals=np.array([h["val_loss"] for h in history])
    return {"optimizer":name,"lr":lr,"batch_size":batch,"seed":seed,
            "epochs_to_target":reached if reached is not None else np.nan,
            "final_val_mse":history[-1]["val_loss"],"test_RMSE":rmse,"test_MAE":mae,
            "loss_std_last10":float(np.std(vals[-10:])),"history":history}

Q4_LRS=[1e-4,1e-3,1e-2]; Q4_BATCHES=[32,128]; Q4_EPOCHS=20
q4_runs=[]
for name in ["SGD","Momentum","RMSProp","Adam"]:
    for lr in Q4_LRS:
        for batch in Q4_BATCHES:
            for seed in SEED_LIST:
                print("Q4:",name,lr,batch,seed)
                q4_runs.append(q4_run(name,lr,batch,seed,Q4_EPOCHS))
q4_df=pd.DataFrame([{k:v for k,v in r.items() if k!="history"} for r in q4_runs])
q4_summary=q4_df.groupby(["optimizer","lr","batch_size"]).agg(
    target_epoch_mean=("epochs_to_target","mean"),RMSE_mean=("test_RMSE","mean"),
    RMSE_std=("test_RMSE","std"),MAE_mean=("test_MAE","mean"),
    final_val_mse_mean=("final_val_mse","mean"),loss_stability=("loss_std_last10","mean")).reset_index()
display(q4_summary.round(4))
q4_df.to_csv(OUT/"Q4_runs.csv",index=False); q4_summary.to_csv(OUT/"Q4_summary.csv",index=False)

In [ ]:
# Q4 — plot optimizer convergence for each optimizer at a representative setting
fig,ax=plt.subplots(figsize=(9,5))
for name in ["SGD","Momentum","RMSProp","Adam"]:
    rr=[r for r in q4_runs if r["optimizer"]==name and r["lr"]==1e-3 and r["batch_size"]==32]
    m=min(len(r["history"]) for r in rr)
    curve=np.mean([[r["history"][i]["val_loss"] for i in range(m)] for r in rr],axis=0)
    ax.plot(np.arange(1,m+1),curve,label=name)
ax.axhline(.35,color="black",linestyle=":",label="target validation MSE")
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation MSE (standardized target)")
ax.set_title("Q4 optimizer convergence (lr=0.001, batch=32)"); ax.grid(True); ax.legend()
plt.savefig(OUT/"Q4_optimizer_convergence.png",dpi=160); plt.show()

### Q4 nightly training budget and interpretation
This notebook defines the nightly budget as **20 epochs**. The target validation MSE is 0.35 in standardized target units. If a configuration never reaches it, `epochs_to_target` is missing/NaN. Compare convergence reliability across three seeds, final test RMSE/MAE in original Wh units, and the last-10-epoch validation-loss standard deviation. A lower standard deviation is a simple stability indicator, not a complete proof of stability.

# Q5 — Derive a convolutional network from a design principle (7 marks)

Dataset: Fashion-MNIST (60,000 training and 10,000 test grayscale 28×28 images). Use a fixed train/validation/test split from the official training set plus official test set.

Design principle: a convolutional filter examines a local neighbourhood and the same weights are reused at every spatial location.

For input `H×W`, kernel `K`, stride `S`, padding `P`, dilation 1:
`H_out = floor((H + 2P − K)/S) + 1`, similarly for width.
Convolution parameters: `(K_h K_w C_in + 1) C_out`.
A flattened fully connected first layer with `N_in` inputs and `N_out` outputs has `(N_in+1)N_out` parameters.

In [ ]:
# Q5 — Fashion-MNIST data
tfm=T.ToTensor()
fashion_train=torchvision.datasets.FashionMNIST("/content/data",train=True,download=True,transform=tfm)
fashion_test=torchvision.datasets.FashionMNIST("/content/data",train=False,download=True,transform=tfm)
# Deterministic 45k/5k split from training set; use official test set for final test.
g=torch.Generator().manual_seed(42)
perm=torch.randperm(len(fashion_train),generator=g).tolist()
train_idx,val_idx=perm[:45000],perm[45000:50000]
fashion_tr=Subset(fashion_train,train_idx); fashion_val=Subset(fashion_train,val_idx)
print("Fashion-MNIST train/val/test:",len(fashion_tr),len(fashion_val),len(fashion_test))

In [ ]:
# Q5 — custom FC and CNN, same loaders and training budget
def loaders_for_q5(seed=42,batch=128):
    gen=torch.Generator().manual_seed(seed)
    return (DataLoader(fashion_tr,batch_size=batch,shuffle=True,generator=gen,num_workers=2),
            DataLoader(fashion_val,batch_size=256,shuffle=False,num_workers=2),
            DataLoader(fashion_test,batch_size=256,shuffle=False,num_workers=2))

class Q5FC(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Flatten(),nn.Linear(784,256),nn.ReLU(),nn.Linear(256,10))
    def forward(self,x): return self.net(x)

class Q5CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1=nn.Conv2d(1,16,3,padding=1)
        self.conv2=nn.Conv2d(16,32,3,padding=1)
        self.pool=nn.MaxPool2d(2)
        self.fc=nn.Linear(32*7*7,10)
    def forward(self,x):
        x=self.pool(F.relu(self.conv1(x)))
        x=self.pool(F.relu(self.conv2(x)))
        return self.fc(torch.flatten(x,1))

def eval_classifier(model,loader):
    model.eval(); ys=[]; ps=[]
    with torch.no_grad():
        for x,y in loader:
            z=model(x.to(DEVICE)); ps.extend(z.argmax(1).cpu().numpy()); ys.extend(y.numpy())
    return accuracy_score(ys,ps)

def train_image_model(model,train_loader,val_loader,epochs=5):
    model=model.to(DEVICE); opt=torch.optim.Adam(model.parameters(),lr=1e-3)
    lossfn=nn.CrossEntropyLoss(); hist=[]
    start=time.perf_counter()
    for ep in range(epochs):
        model.train(); losses=[]
        for x,y in train_loader:
            x=x.to(DEVICE); y=y.to(DEVICE)
            opt.zero_grad(); loss=lossfn(model(x),y); loss.backward(); opt.step(); losses.append(loss.item())
        hist.append({"epoch":ep+1,"train_loss":float(np.mean(losses)),"val_accuracy":eval_classifier(model,val_loader)})
    elapsed=time.perf_counter()-start
    return model,hist,elapsed

def count_params(model): return sum(p.numel() for p in model.parameters() if p.requires_grad)

q5_runs=[]; q5_models={}
for arch,Cls in [("FC",Q5FC),("CNN",Q5CNN)]:
    for seed in SEED_LIST:
        torch.manual_seed(seed); np.random.seed(seed)
        trl,vl, tel=loaders_for_q5(seed)
        model,hist,elapsed=train_image_model(Cls(),trl,vl,epochs=5)
        test_acc=eval_classifier(model,tel)
        q5_models[(arch,seed)]=model
        q5_runs.append({"architecture":arch,"seed":seed,"parameters":count_params(model),
                        "train_seconds":elapsed,"test_accuracy":test_acc,"history":hist})
q5_df=pd.DataFrame([{k:v for k,v in r.items() if k!="history"} for r in q5_runs])
display(q5_df.groupby("architecture").agg(
    params=("parameters","mean"),training_seconds=("train_seconds","mean"),
    test_accuracy_mean=("test_accuracy","mean"),test_accuracy_std=("test_accuracy","std")).reset_index().round(4))
q5_df.to_csv(OUT/"Q5_comparison.csv",index=False)

In [ ]:
# Q5 — translation test, filters and feature maps
def shifted_batch(x,tx=2,ty=2):
    theta=torch.tensor([[[1,0,tx/14],[0,1,ty/14]]],dtype=x.dtype,device=x.device).repeat(len(x),1,1)
    grid=F.affine_grid(theta,x.size(),align_corners=False)
    return F.grid_sample(x,grid,align_corners=False)

def accuracy_on_shift(model,loader):
    model.eval(); ys=[]; ps=[]
    with torch.no_grad():
        for x,y in loader:
            x=shifted_batch(x.to(DEVICE),2,2)
            ps.extend(model(x).argmax(1).cpu().numpy()); ys.extend(y.numpy())
    return accuracy_score(ys,ps)

trl,vl,tel=loaders_for_q5(42)
translation_rows=[]
for arch in ["FC","CNN"]:
    m=q5_models[(arch,42)]
    clean=eval_classifier(m,tel); shifted=accuracy_on_shift(m,tel)
    translation_rows.append({"architecture":arch,"clean_accuracy":clean,
                             "shifted_accuracy":shifted,"accuracy_drop":clean-shifted})
display(pd.DataFrame(translation_rows).round(4))

cnn=q5_models[("CNN",42)]
filters=cnn.conv1.weight.detach().cpu()
fig,axs=plt.subplots(2,8,figsize=(12,3))
for i,ax in enumerate(axs.flat):
    ax.imshow(filters[i,0],cmap="gray"); ax.axis("off"); ax.set_title(str(i))
plt.suptitle("Q5 learned first-layer filters"); plt.tight_layout()
plt.savefig(OUT/"Q5_filters.png",dpi=160); plt.show()

x0,y0=next(iter(tel)); sample=x0[:1].to(DEVICE)
with torch.no_grad():
    fmap=F.relu(cnn.conv1(sample)).cpu()[0]
fig,axs=plt.subplots(2,8,figsize=(12,3))
for i,ax in enumerate(axs.flat):
    ax.imshow(fmap[i],cmap="viridis"); ax.axis("off"); ax.set_title(str(i))
plt.suptitle(f"Q5 first-layer feature maps (label={int(y0[0])})"); plt.tight_layout()
plt.savefig(OUT/"Q5_feature_maps.png",dpi=160); plt.show()

### Q5 interpretation
The same learned kernel is applied at every image position. Consequently, a local pattern can activate the same filter even when it appears at a different location. Weight sharing reduces parameters relative to a dense layer and gives translation equivariance in the convolutional feature map; pooling can add tolerance to small shifts, although neither property guarantees perfect translation invariance. Compare the measured translation drops rather than claiming CNNs are always invariant.

# Q6 — Architecture design: capacity, cost and deployment boundary (8 marks)

Dataset: CIFAR-10, using four selected classes (`cat`, `deer`, `dog`, `horse`) as a small-image proxy for on-device camera-trap classification. It is not a wildlife dataset and does not contain elephant/human/empty labels.

Three student-designed CNNs vary depth, width and input resolution. They are not copied from VGG, ResNet or MobileNet.

For convolution layer `l`:
- Parameters = `(K² C_in + 1) C_out`
- MACs = `H_out × W_out × C_out × K² × C_in`
- Receptive field recurrence (stride 1): `r_l = r_(l−1) + (K−1)j_(l−1)` and `j_l = j_(l−1)`; pooling changes the jump `j`.
- Doubling width approximately quadruples convolution MACs when both input and output channels double. Doubling spatial resolution approximately quadruples MACs. Doubling depth approximately doubles cost only when the added layers have similar widths and spatial dimensions; the measured family is needed to compare.

In [ ]:
# Q6 — CIFAR-10 four-class data and three architecture definitions
cifar_tfm=T.Compose([T.Resize((32,32)),T.ToTensor()])
cifar_train=torchvision.datasets.CIFAR10("/content/data",train=True,download=True,transform=cifar_tfm)
cifar_test=torchvision.datasets.CIFAR10("/content/data",train=False,download=True,transform=cifar_tfm)
chosen=[3,4,5,7]  # cat, deer, dog, horse
class_names=["cat","deer","dog","horse"]
def filtered_indices(ds):
    return [i for i,(_,y) in enumerate(ds) if y in chosen]
train_all=filtered_indices(cifar_train); test_all=filtered_indices(cifar_test)
rng=np.random.default_rng(42); rng.shuffle(train_all)
train_all=train_all[:6000]; val_idx=train_all[5000:]; train_idx=train_all[:5000]
test_all=test_all[:1500]
class RemapSubset(torch.utils.data.Dataset):
    def __init__(self,base,indices): self.base=base; self.indices=indices
    def __len__(self): return len(self.indices)
    def __getitem__(self,i):
        x,y=self.base[self.indices[i]]
        return x,chosen.index(y)
q6_train=RemapSubset(cifar_train,train_idx); q6_val=RemapSubset(cifar_train,val_idx); q6_test=RemapSubset(cifar_test,test_all)
print("Q6 four classes:",class_names,"train/val/test:",len(q6_train),len(q6_val),len(q6_test))

Q6_ARCHS={
 "Small": {"channels":[16,32],"resolution":32},
 "Medium":{"channels":[24,48,64],"resolution":32},
 "Wide_HighRes":{"channels":[32,64,96],"resolution":48}
}
class CustomCNN(nn.Module):
    def __init__(self,channels,num_classes=4,resolution=32):
        super().__init__(); self.resolution=resolution
        layers=[]; cin=3
        for i,cout in enumerate(channels):
            layers += [nn.Conv2d(cin,cout,3,padding=1),nn.ReLU()]
            if i%2==1: layers += [nn.MaxPool2d(2)]
            cin=cout
        self.features=nn.Sequential(*layers)
        with torch.no_grad():
            dummy=torch.zeros(1,3,resolution,resolution)
            n=self.features(dummy).numel()
        self.classifier=nn.Linear(n,num_classes)
    def forward(self,x):
        if x.shape[-1]!=self.resolution:
            x=F.interpolate(x,size=(self.resolution,self.resolution),mode="bilinear",align_corners=False)
        return self.classifier(torch.flatten(self.features(x),1))

def q6_loaders(seed):
    gen=torch.Generator().manual_seed(seed)
    return (DataLoader(q6_train,batch_size=128,shuffle=True,generator=gen,num_workers=2),
            DataLoader(q6_val,batch_size=256,shuffle=False,num_workers=2),
            DataLoader(q6_test,batch_size=256,shuffle=False,num_workers=2))

In [ ]:
# Q6 — hand-check every layer output shape, parameters and MACs
def conv_shape(n,k=3,s=1,p=1): return (n+2*p-k)//s+1

def architecture_hand_table(channels,resolution,num_classes=4):
    h=w=resolution; cin=3; rows=[]; total_macs=0; total_params=0
    rf=1; jump=1
    for i,cout in enumerate(channels):
        hout=conv_shape(h); wout=conv_shape(w)
        params=(3*3*cin+1)*cout
        layer_macs=hout*wout*cout*3*3*cin
        rf=rf+2*jump
        rows.append({"layer":f"conv{i+1}","output_shape":f"{cout}x{hout}x{wout}",
                     "parameters":params,"MACs":layer_macs,"receptive_field":rf})
        total_params+=params; total_macs+=layer_macs
        h,w,cin=hout,wout,cout
        if i%2==1:
            h//=2; w//=2; jump*=2; rf=rf+jump
            rows.append({"layer":f"maxpool_after_conv{i+1}",
                         "output_shape":f"{cin}x{h}x{w}",
                         "parameters":0,"MACs":0,"receptive_field":rf})
    flat=channels[-1]*h*w
    fc_params=(flat+1)*num_classes
    fc_macs=flat*num_classes
    rows.append({"layer":"classifier","output_shape":str(num_classes),
                 "parameters":fc_params,"MACs":fc_macs,"receptive_field":rf})
    total_params+=fc_params; total_macs+=fc_macs
    return pd.DataFrame(rows),total_params,total_macs

for name,spec in Q6_ARCHS.items():
    tab,hand_params,hand_macs=architecture_hand_table(
        spec["channels"],spec["resolution"])
    check_model=CustomCNN(spec["channels"],resolution=spec["resolution"])
    framework_params=count_params(check_model)
    print("\\n",name,"hand calculation vs framework summary")
    display(tab)
    print("Hand parameters:",hand_params,
          "| Framework parameters:",framework_params,
          "| Match:",hand_params==framework_params)
    print("Hand MACs per image (conv + classifier):",hand_macs)

In [ ]:
# Q6 — train 3 architecture sizes across 3 seeds; record params, MACs, file size, latency, accuracy
def q6_eval(model,loader,resolution):
    model.eval(); ys=[]; ps=[]
    with torch.no_grad():
        for x,y in loader:
            x=F.interpolate(x.to(DEVICE),size=(resolution,resolution),mode="bilinear",align_corners=False)
            ps.extend(model(x).argmax(1).cpu().numpy()); ys.extend(y.numpy())
    return accuracy_score(ys,ps)

def estimate_macs(model,resolution):
    total={"macs":0}
    def hook(m,inp,out):
        if isinstance(m,nn.Conv2d):
            b,co,h,w=out.shape; kh,kw=m.kernel_size; ci=m.in_channels
            total["macs"]+=b*h*w*co*kh*kw*ci
        elif isinstance(m,nn.Linear):
            total["macs"]+=out.shape[0]*m.in_features*m.out_features
    hooks=[m.register_forward_hook(hook) for m in model.modules() if isinstance(m,(nn.Conv2d,nn.Linear))]
    with torch.no_grad(): model(torch.zeros(1,3,resolution,resolution,device=DEVICE))
    for h in hooks: h.remove()
    return total["macs"]

q6_rows=[]; q6_models={}
for name,spec in Q6_ARCHS.items():
    for seed in SEED_LIST:
        torch.manual_seed(seed); np.random.seed(seed)
        trl,vall,tel=q6_loaders(seed)
        model=CustomCNN(spec["channels"],resolution=spec["resolution"]).to(DEVICE)
        opt=torch.optim.Adam(model.parameters(),lr=1e-3); lossfn=nn.CrossEntropyLoss()
        start=time.perf_counter()
        for ep in range(5):
            model.train()
            for x,y in trl:
                x=F.interpolate(x.to(DEVICE),size=(spec["resolution"],spec["resolution"]),mode="bilinear",align_corners=False)
                y=y.to(DEVICE); opt.zero_grad(); loss=lossfn(model(x),y); loss.backward(); opt.step()
        train_sec=time.perf_counter()-start
        # latency per image, batch=1, warm-up then repeated timing
        model.eval(); sample=torch.zeros(1,3,spec["resolution"],spec["resolution"],device=DEVICE)
        with torch.no_grad():
            for _ in range(10): model(sample)
            t0=time.perf_counter()
            for _ in range(100): model(sample)
            latency=(time.perf_counter()-t0)/100
        test_acc=q6_eval(model,tel,spec["resolution"])
        params=count_params(model); macs=estimate_macs(model,spec["resolution"])
        model_path=OUT/f"Q6_{name}_seed{seed}.pt"; torch.save(model.state_dict(),model_path)
        q6_models[(name,seed)]=model
        q6_rows.append({"architecture":name,"seed":seed,"depth":len(spec["channels"]),
                        "width_max":max(spec["channels"]),"resolution":spec["resolution"],
                        "parameters":params,"MACs_per_image":macs,
                        "model_size_MB":model_path.stat().st_size/1e6,
                        "inference_ms_per_image":latency*1000,
                        "train_seconds":train_sec,"test_accuracy":test_acc})
        print("Q6 complete:",name,seed,"accuracy",round(test_acc,4))
q6_df=pd.DataFrame(q6_rows)
q6_summary=q6_df.groupby("architecture").agg(
    parameters=("parameters","mean"),MACs=("MACs_per_image","mean"),
    model_size_MB=("model_size_MB","mean"),inference_ms=("inference_ms_per_image","mean"),
    accuracy_mean=("test_accuracy","mean"),accuracy_std=("test_accuracy","std")).reset_index()
display(q6_summary.round(4))
q6_df.to_csv(OUT/"Q6_runs.csv",index=False); q6_summary.to_csv(OUT/"Q6_summary.csv",index=False)

### Q6 deployment decision
This notebook uses an illustrative, explicitly chosen deployment budget of **5 MB model file size and 10 ms per image** on the measured Colab runtime. Colab latency is not equivalent to the target camera-trap hardware; the budget is an experimental decision boundary, not a claim about a real device. Select the highest-accuracy measured architecture within both limits, then report the accuracy sacrificed relative to the largest model. If no model meets the budget, state that and revise the budget or architecture transparently.

# Q7 — Diagnose, redesign and validate (13 marks)

Q7 must use evidence from Q1–Q6. This notebook selects the Q1 eight-layer sigmoid model as the candidate limitation **only if the measured Q1 results show materially weaker early-layer gradients or worse validation/test behaviour than the eight-layer ReLU model**. The redesign is a student-designed eight-layer ReLU network with He initialization. It is not a swap to a famous architecture.

The objective is to improve early-layer gradient flow and validation performance without changing dataset split, seeds, optimizer or epoch budget.

In [ ]:
# Q7 — compare original deep sigmoid vs redesigned deep ReLU+He, same seeds/data/budget
# Q1MLP already uses Xavier for sigmoid and He for ReLU.
q7_rows=[]; q7_histories={}
for design,act in [("Original_8L_Sigmoid","sigmoid"),("Redesigned_8L_ReLU_He","relu")]:
    for seed in SEED_LIST:
        r,_=q1_run("Deep_8L",act,seed)
        q7_histories[(design,seed)]=r["history"]
        # Early-layer gradient magnitude averaged over first 10 epochs, using a matched
        # fresh training pass with gradients recorded under the same configuration.
        # q1_run records these separately in Q1; retrieve matching records.
        match=q1_grad_df[(q1_grad_df.config=="Deep_8L") &
                         (q1_grad_df.activation==act) & (q1_grad_df.seed==seed)]
        early=match[match.layer==1].gradient_norm.mean()
        q7_rows.append({"design":design,"seed":seed,"test_accuracy":r["test_accuracy"],
                        "test_precision":r["test_precision"],"test_recall":r["test_recall"],
                        "final_train_loss":r["final_train_loss"],"final_val_loss":r["final_val_loss"],
                        "early_layer_gradient_norm":early})
q7_df=pd.DataFrame(q7_rows)
display(q7_df.groupby("design").agg(
    test_accuracy_mean=("test_accuracy","mean"),test_accuracy_std=("test_accuracy","std"),
    validation_loss_mean=("final_val_loss","mean"),
    early_gradient_mean=("early_layer_gradient_norm","mean")).reset_index().round(5))
q7_df.to_csv(OUT/"Q7_redesign_comparison.csv",index=False)

fig,ax=plt.subplots(figsize=(8,4))
for design in ["Original_8L_Sigmoid","Redesigned_8L_ReLU_He"]:
    hs=[q7_histories[(design,s)] for s in SEED_LIST]
    m=min(len(h) for h in hs)
    curve=np.mean([[h[i]["val_loss"] for i in range(m)] for h in hs],axis=0)
    ax.plot(curve,label=design)
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation BCE"); ax.set_title("Q7 original vs redesigned"); ax.grid(True); ax.legend()
plt.savefig(OUT/"Q7_redesign_validation_loss.png",dpi=160); plt.show()

### Q7 required written defence (complete after running)

**A. Diagnose (2):** identify the specific layer/epoch and metric from Q1 (gradient norms, loss or accuracy). State whether the cause is activation saturation/gradient attenuation, initialization, capacity, data or optimization. Do not use generic statements.

**B. Objective (2):** state a numerical target based on the original measured values, such as increasing the mean first-layer gradient norm and/or reducing final validation loss while keeping eight hidden layers and the same epoch budget.

**C. Revised design (4):** eight hidden layers, ReLU hidden units and He initialization. ReLU derivative is 1 on active units, and He initialization scales weights for ReLU networks. Trade-off: inactive units can receive zero gradient; monitor the dead-ReLU fraction.

**D. Prediction (1):** predict improved early-layer gradient norms and a lower validation loss than the original sigmoid model.

**E. Validate (2):** compare the table above using the same dataset, train/validation/test split, three seeds, optimizer and epoch budget.

**F. Defend (2):** report whether the prediction was supported, the magnitude of improvement, dead-unit/compute trade-offs, when sigmoid might still be useful, and whether the objective was met. A failed hypothesis is still reportable if the evidence and explanation are correct.

# Final export — raw evidence and figures

The assignment requires source code/notebooks, training logs, random seeds and hyperparameters, raw experimental data, learning curves and model-specific evidence. This cell creates a ZIP containing the CSVs and figures generated above. The `.ipynb` itself is the source notebook.

In [ ]:
# Export all generated evidence files into one ZIP
import shutil
zip_out="/content/CO2_Assignment2_Evidence.zip"
with zipfile.ZipFile(zip_out,"w",zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file(): z.write(p,arcname=str(p.relative_to(OUT)))
print("Evidence ZIP:",zip_out)
print("Files:")
for p in sorted(OUT.iterdir()):
    print(p.name)

## Final submission checks

Before submitting:
1. Run the notebook from top to bottom in a fresh Colab runtime.
2. Check that all downloads completed and every experiment finished.
3. Inspect all generated tables and plots; replace any failed/NaN run with a documented rerun.
4. Write the question-wise interpretations using the actual outputs—do not copy generic predictions as observed findings.
5. State dataset provenance, sample counts, splits, seeds, hardware, epochs and hyperparameters in the report.
6. Remember the proxy-dataset limitations stated at the top. If your faculty requires literal plant-leaf or wildlife datasets, substitute an approved dataset and rerun the corresponding question.